# Track-A v1.2 — MASTER K2 (v6)

Two-GPU worker. Acquires canonical K1 G1A, runs CAL-MNV4-LOGITS + CAL-MNV4-FEATURE, waits for the sealed control plane, then executes K2 science.

Use **Kaggle T4 x2**, **Internet ON**, and **Save Version -> Save & Run All / Batch**.

**Frozen science SHA:** `9a72e9466a9a3e7429e0e36a028edac662f83146`  
**Pinned operator runtime:** `280743cf619d47b093944de6ea63be62dcb8f7ae`  
**Runtime branch:** `ops-tracka-kaggle-master-runtime-v6-280743c`

Attach the frozen CropCop V1 dataset. K1's canonical private G1A dataset must be shared with this account as **Can view** before K2 can consume the READY handoff. Required secrets: `KAGGLE_USERNAME`, `KAGGLE_KEY`, `CROPCOP_GITHUB_TOKEN`.

v6 serializes duplicate notebook launches per account so only one K2 orchestration can own the session; duplicate launches follow the canonical owner result instead of racing.


In [ ]:
from pathlib import Path
import os

V1 = Path('/kaggle/input/datasets/ranamuhammadahmed6/cropcop-finalized-v8-11-2026-1/CropCop_Final_v1')
if V1.is_dir():
    preferred = {
        'CROPCOP_MANIFEST': V1 / 'audit' / 'final_manifest.csv',
        'CROPCOP_CLASS_MAP': V1 / 'audit' / 'class_to_idx.json',
        'CROPCOP_IMAGE_ROOT': V1 / 'dataset',
    }
    for name, path in preferred.items():
        if path.exists():
            os.environ.setdefault(name, str(path))
    print('Preferred frozen V1 mount detected:', V1)
else:
    print('Preferred V1 mount prefix not present; bounded verified resolver will be used.')
# Optional explicit account binding:
# os.environ['CROPCOP_EXPECTED_KAGGLE_USERNAME'] = 'your-k2-username'


In [ ]:
from pathlib import Path
import shutil, subprocess, sys
OPS_RUNTIME_SHA = '280743cf619d47b093944de6ea63be62dcb8f7ae'
OPS_RUNTIME_BRANCH = 'ops-tracka-kaggle-master-runtime-v6-280743c'
OPS_ROOT = Path('/kaggle/working/cropcop-tracka-master-runtime')
if OPS_ROOT.exists():
    shutil.rmtree(OPS_ROOT)
subprocess.run(['git','clone','--quiet','--depth','1','--branch',OPS_RUNTIME_BRANCH,'https://github.com/rana-m-ahmed/ResearchWork-CropCop.git',str(OPS_ROOT)], check=True)
head = subprocess.check_output(['git','rev-parse','HEAD'], cwd=OPS_ROOT, text=True).strip()
dirty = subprocess.check_output(['git','status','--porcelain'], cwd=OPS_ROOT, text=True).strip()
if head != OPS_RUNTIME_SHA:
    raise RuntimeError(f'operator runtime SHA mismatch: expected {OPS_RUNTIME_SHA}, got {head}')
if dirty:
    raise RuntimeError(f'operator runtime checkout is dirty: {dirty}')
print('Pinned Track-A master runtime:', head)


In [ ]:
launcher = OPS_ROOT / 'journal_extension/kaggle/tracka_v12_ops/master_launch_guard_v6.py'
cp = subprocess.run([sys.executable, '-u', str(launcher), 'K2'], cwd=launcher.parent)
if cp.returncode == 0:
    print('K2 master: TERMINAL PASS for this account queue.')
elif cp.returncode == 2:
    print('K2 master: controlled dependency/session/publication continuation. Rerun THIS SAME notebook as a fresh Batch version.')
else:
    raise RuntimeError(f'K2 master requires investigation; rc={cp.returncode}')


Recovery rule: for `rc=2`, rerun this same K2 notebook in a fresh Batch session. Hard failures remain fail-closed.
